# Project 2: Bayesian optimization

This notebook is the starting point of the project; the subject (`projects.pdf`, section 2)
gives the context and the grading. You hand in this notebook, completed: it must run from
top to bottom in less than 15 minutes on a laptop.

The project has three guided tasks, then a part where you choose how to go further:

1. **GP-UCB**: write the loop of Bayesian optimization with the Gaussian process given below.
2. **Against random search**: compare the two methods on $20$ seeds, and choose what to
   recommend at the end.
3. **Exploration**: study the parameter $\beta$ of GP-UCB.
4. **Going further**: a direction of your choice (see the end of the notebook).

Each task ends with a figure or a table and a short written answer (a few sentences) in the
cell "Answer". Everything outside the cells marked `...  # TODO` is given and working.

This notebook needs the version of `seqlearn` that contains `projects/`: if the imports below
fail, download the setup archive again from the course page and replace the folder
`seqlearn/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from seqlearn.plotting import COLORS, MUTED, set_style
from seqlearn.projects.bayesopt import BUDGET, DIM, BlackBox, decode

set_style()

## 0. The black box

`BlackBox(rng).evaluate(x)` trains a small neural network (one hidden layer, in numpy) on a
classification problem and returns its validation error. The point $x \in [0, 1]^4$ encodes
$4$ hyperparameters: learning rate, weight decay, number of hidden units, batch size
(`decode(x)` gives their values). Each evaluation is **noisy**: two trainings with the same
$x$ give different errors. The budget is $40$ evaluations, as if each one took hours.

In [ ]:
box = BlackBox(rng=0)
x = np.array([0.7, 0.1, 0.8, 0.2])
print(decode(x))
print("5 evaluations of the same x:", [round(box.evaluate(x), 3) for _ in range(5)])

## Given: the Gaussian process, random search and the evaluation

`gp_posterior(X, y, candidates)` returns the posterior mean $\mu(x)$ and standard deviation
$s(x)$ of a Gaussian process with a squared exponential kernel of length scale `length`,
fitted on the evaluated points `X` (shape $(n, 4)$) and their errors `y`, at each row of
`candidates`. The formulas are in the subject; the mean is a kernel ridge regression.

A method is a function `method(box, rng)` that spends the budget of `box` and returns the
point it **recommends**. Its quality is the **true error** of this point, estimated by
`true_error`, the mean of $10$ new trainings (not counted in the budget). It is not the best
error observed during the search, which is optimistic because of the noise.

In [ ]:
def gp_posterior(X, y, candidates, length=0.25, noise=0.03):
    X, y = np.asarray(X), np.asarray(y)
    m, var = y.mean(), max(y.var(), 1e-4)

    def kernel(A, B):
        d2 = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2)
        return var * np.exp(-d2 / (2 * length ** 2))

    L = np.linalg.cholesky(kernel(X, X) + noise ** 2 * np.eye(len(X)))
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y - m))
    k = kernel(candidates, X)
    mean = m + k @ alpha
    v = np.linalg.solve(L, k.T)
    sd = np.sqrt(np.maximum(var - (v ** 2).sum(axis=0), 1e-12))
    return mean, sd


def random_search(box, rng):
    X = rng.random((BUDGET, DIM))
    y = [box.evaluate(x) for x in X]
    return X[int(np.argmin(y))]


def true_error(x, seed):
    fresh = BlackBox(rng=10_000 + seed)
    return np.mean([fresh.evaluate(x) for _ in range(10)])


def compare(methods, n_seeds=20):
    """True errors of the points recommended by each method, on seeds 0 to n_seeds - 1."""
    results = {}
    for name, method in methods.items():
        errors = []
        for seed in range(n_seeds):
            box = BlackBox(rng=seed)
            x = method(box, np.random.default_rng(seed))
            assert box.n_evaluations <= BUDGET, f"{name} used {box.n_evaluations} evaluations"
            errors.append(true_error(x, seed))
        results[name] = np.array(errors)
        se = results[name].std() / np.sqrt(n_seeds)
        print(f"{name:<32} true error {results[name].mean():.3f} +- {2 * se:.3f}")
    return results


def plot_results(results, title):
    fig, ax = plt.subplots(figsize=(7, 0.6 + 0.5 * len(results)))
    for k, (name, errors) in enumerate(results.items()):
        ax.plot(errors, np.full(len(errors), k) + np.random.default_rng(k).uniform(-0.15, 0.15, len(errors)),
                "o", color=COLORS[k % len(COLORS)], alpha=0.5, markersize=4)
        se = errors.std() / np.sqrt(len(errors))
        ax.errorbar(errors.mean(), k, xerr=2 * se, fmt="D", color="black", capsize=4)
    ax.set_yticks(range(len(results)), list(results))
    ax.set(xlabel="true error of the recommended point (one dot per seed; mean +- 2 s.e.)", title=title)
    plt.show()

## Task 1: GP-UCB

Complete `gp_ucb`. Start with `n_init` random points; then, until the budget is spent,
fit the Gaussian process on the points evaluated so far, draw $2000$ random candidates in
$[0, 1]^4$, and evaluate the candidate that minimizes $\mu(x) - \beta\, s(x)$ (we minimize an
error: the optimistic value is the *lowest* plausible one). At the end, recommend the
evaluated point with the lowest posterior mean.

Then run it on seed $0$ and plot the error of each evaluation, for GP-UCB and for random
search.

**Expected**: the figure, and an answer to: how do the evaluations of GP-UCB evolve along
the budget, compared with random search? Why does GP-UCB recommend the lowest posterior mean
rather than the lowest observed error?

**To read**: slides of session 2 (optimism) and session 3 (ridge regression); Srinivas,
Krause, Kakade and Seeger, *Gaussian process optimization in the bandit setting*, ICML 2010
(sections 1 and 2); Rasmussen and Williams, *Gaussian Processes for Machine Learning*, MIT
Press 2006, chapter 2 (free online).

In [ ]:
def gp_ucb(box, rng, beta=2.0, n_init=5, length=0.25, history=None):
    X = list(rng.random((n_init, DIM)))
    y = [box.evaluate(x) for x in X]
    ...  # TODO
    if history is not None:
        history.extend(y)
    return best

In [ ]:
history_gp, history_rs = [], []
gp_ucb(BlackBox(rng=0), np.random.default_rng(0), history=history_gp)
box = BlackBox(rng=0)
X_rs = np.random.default_rng(0).random((BUDGET, DIM))
history_rs = [box.evaluate(x) for x in X_rs]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(range(1, BUDGET + 1), history_rs, "o-", color=COLORS[1], markersize=4, linewidth=1, label="random search")
ax.plot(range(1, BUDGET + 1), history_gp, "o-", color=COLORS[0], markersize=4, linewidth=1, label="GP-UCB")
ax.set(xlabel="evaluation", ylabel="observed validation error", title="Task 1: one run (seed 0)")
ax.legend()
plt.show()

**Answer.** *Write your answer here.*

## Task 2: against random search

Compare GP-UCB and random search on $20$ seeds with `compare`, and plot the results with
`plot_results`. Then add a variant of GP-UCB that recommends the evaluated point with the
lowest **observed** error (write it with a small change of `gp_ucb`, for instance an argument
`recommend`).

**Expected**: the figure, and an answer to: is GP-UCB better than random search, by how
much, and is the difference significant given the error bars? Which recommendation rule is
better?

In [ ]:
...  # TODO
plot_results(results, "Task 2")

**Answer.** *Write your answer here.*

## Task 3: exploration

Compare GP-UCB with $\beta \in \{0, 2, 8, 32\}$ on $20$ seeds.

**Expected**: the figure, and an answer to: what happens with $\beta = 0$ (no exploration
bonus) and with a large $\beta$? Relate it to the exploration-exploitation trade-off of the
bandits.

In [ ]:
...  # TODO
plot_results(results3, "Task 3")

**Answer.** *Write your answer here.*

## Going further

Choose one direction, among these suggestions or your own idea: state the question you ask,
then answer it as in the tasks (code, a figure, a short answer). One well-studied direction
is better than several superficial ones.

- The length scale of the kernel: fixed at $0.25$ here; try other values, or fit it by
  maximizing the marginal likelihood (Rasmussen and Williams, section 5.4).
- Expected improvement, or Thompson sampling for Gaussian processes, instead of GP-UCB.
- Cheap evaluations: `box.evaluate(x, epochs=5)` costs $5/30$ of a training (attribute
  `box.cost`); does sequential halving over random configurations with increasing numbers of
  epochs (Hyperband, Li et al., JMLR 2018) beat GP-UCB at equal cost?